# Model Compression via Ensembling

Ensembling belongs in a compression library for a reason that is not obvious, because on
its face it is the *opposite* of compression: running five models costs five times as
much as running one.

It earns its place through three specific manoeuvres:

1. **Several small models can beat one large one** at equal or lower total parameter
   count — when their errors are decorrelated.
2. **Weight averaging (model soups)** collapses an ensemble into a *single* model with
   exactly the original inference cost, keeping some of the gain.
3. **Ensemble-then-distil** uses an expensive ensemble as a teacher for one cheap
   student — see [Distillation](knowledge-distillation.ipynb).

The quantity that decides whether any of this works is **error correlation**, and it is
the thing most discussions of ensembling leave unmeasured. This notebook measures it.

## 1. What & Why

Averaging predictions reduces variance. If you average `N` predictors whose errors are
independent with variance `σ²`, the ensemble's error variance is `σ²/N` — the classic
result. The catch, and the entire practical subject, is that **real models trained on the
same data are not independent**. They share an architecture, a dataset and usually an
initialisation scheme, so their errors are correlated, and correlated errors do not
cancel.

The general form, for `N` predictors with pairwise error correlation `ρ`:

```
Var(ensemble) = σ² · [ ρ + (1 − ρ)/N ]
```

Read that carefully, because it is the whole story. As `N → ∞`, the variance does not go
to zero — it goes to `ρσ²`. **Correlation sets a floor that no amount of ensembling can
break through.** At `ρ = 0.9`, an infinite ensemble buys you a 10% variance reduction.

**Reach for ensembling when:**

- You need the last point or two of accuracy and can afford the inference cost.
- You need **calibration** or uncertainty estimates — this is deep ensembles' real
  strength and it is not a compression argument at all.
- You are building a teacher for distillation. An ensemble is the cheapest way to make a
  better teacher than any model you have.

**Don't when:** you are latency-bound, or you were going to ensemble `N` checkpoints from
the same run without checking whether their errors differ at all.

## 2. Mental Model

**A panel of specialists versus one generalist — but only if the panel disagrees.**

Convening five experts is worth more than consulting one, provided they were trained
differently and make *different* mistakes. Five graduates of the same programme, taught
by the same instructor, from the same textbook, will confidently make the same error
together. You paid five consulting fees for one opinion.

That is what `ρ` measures, and it explains the two headline techniques:

- **Deep ensembles** work because random initialisation and data ordering land the models
  in genuinely different loss basins — different enough that the mistakes differ.
- **Model soups** work for the opposite reason: fine-tuned runs from *the same*
  pretrained checkpoint stay in one basin, so their weights can be averaged directly
  without the network breaking. Same-basin models can be merged; different-basin models
  can only be ensembled at the output.

That single distinction — same basin or not — tells you which technique is available to
you.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Variance reduction** | The mechanism. Averaging `N` decorrelated predictors cuts error variance toward `σ²/N`. |
| **Error correlation `ρ`** | How much the members are wrong *together*. Sets the floor `ρσ²`. The number to measure first. |
| **Diversity** | Anything that decorrelates members: different seeds, data order, augmentation, architecture, hyperparameters, data subsets (bagging). |
| **Deep ensembles** | Independently-trained copies of the same architecture. Strong accuracy *and* the standard baseline for uncertainty. |
| **Snapshot ensembles** | Members collected from a single training run using a cyclic learning rate. Nearly free to train; more correlated, so less effective. |
| **Logit vs probability averaging** | Averaging before or after the softmax. Different operations (geometric vs arithmetic mean), different behaviour under disagreement. |
| **Model soups / weight averaging** | Average the *weights*, not the outputs. Yields one model at 1× inference cost. Requires members in the same loss basin. |
| **Linear mode connectivity** | The property that makes soups possible: two fine-tunes from one checkpoint are joined by a low-loss linear path. |
| **Mixture of Experts** | A *sparse* ensemble: many experts, only `k` run per token, so capacity grows while cost stays flat. See [MoE](../08-architectures/mixture-of-experts.ipynb). |
| **Ensemble distillation** | Train one student on the ensemble's averaged output. Converts an `N×`-cost ensemble into a `1×`-cost model. |
| **Calibration** | Whether stated confidence matches observed accuracy. Ensembles improve it markedly; single models are typically overconfident. |

## 4. Setup

NumPy only. The experiments below are deliberately synthetic because the effect of
interest — the relationship between error correlation and ensemble gain — is obscured by
everything else in a real training run.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — correlation is the whole game

Build ensembles of members with controlled error correlation and measure the actual
reduction against the theoretical `ρ + (1−ρ)/N`.

In [2]:
def correlated_errors(n_samples, N, rho, sigma=1.0):
    '''N predictors whose errors have pairwise correlation rho.

    Built as: shared component (weight sqrt(rho)) + private component (sqrt(1-rho)).
    '''
    shared = rng.standard_normal((n_samples, 1))
    private = rng.standard_normal((n_samples, N))
    return sigma * (np.sqrt(rho) * shared + np.sqrt(1 - rho) * private)

n_samples = 200_000
print(f"{'rho':>6} | " + " ".join(f"N={N:<7}" for N in (2, 4, 8, 32)) + "  (variance vs a single model)")
for rho in (0.0, 0.3, 0.6, 0.9, 0.99):
    row = []
    for N in (2, 4, 8, 32):
        e = correlated_errors(n_samples, N, rho)
        measured = e.mean(axis=1).var() / e[:, 0].var()
        row.append(f"{measured:.3f}   ")
    theory = rho + (1 - rho) / 32
    print(f"{rho:6.2f} | " + " ".join(row) + f"  floor at N=inf: {rho:.2f}")

print("\nAt rho=0 the classic 1/N holds: 32 members give 1/32 the error variance.")
print("At rho=0.9 the same 32 members give 0.90 -- a 10% improvement for 32x the cost.")
print("\nThis is why 'we ensembled 5 checkpoints and it barely helped' is the norm:")
print("checkpoints from one run are highly correlated by construction.")

   rho | N=2       N=4       N=8       N=32       (variance vs a single model)
  0.00 | 0.499    0.250    0.125    0.031     floor at N=inf: 0.00
  0.30 | 0.650    0.475    0.386    0.320     floor at N=inf: 0.30
  0.60 | 0.803    0.702    0.651    0.612     floor at N=inf: 0.60


  0.90 | 0.953    0.927    0.912    0.902     floor at N=inf: 0.90
  0.99 | 0.995    0.993    0.992    0.990     floor at N=inf: 0.99

At rho=0 the classic 1/N holds: 32 members give 1/32 the error variance.
At rho=0.9 the same 32 members give 0.90 -- a 10% improvement for 32x the cost.

This is why 'we ensembled 5 checkpoints and it barely helped' is the norm:
checkpoints from one run are highly correlated by construction.


### Example 2 — several small models vs one big one

The compression claim, stated as a fair test: hold the **total parameter budget** fixed.
Is it better spent on one large model or `N` small ones?

Modelled here as: a bigger model has lower bias (it fits the signal better), while an
ensemble of small models has lower variance. Which wins depends on where the error is
coming from.

In [3]:
def simulate(budget, N, rho, bias_exponent=0.5, noise=1.0):
    '''One model of size `budget/N`, ensembled N times.

    bias^2 falls with model size as size^-bias_exponent; variance is fixed per member
    and reduced by ensembling according to the correlation formula.
    '''
    size = budget / N
    bias_sq = size ** (-bias_exponent)
    var = noise * (rho + (1 - rho) / N)
    return bias_sq + var

BUDGET = 64.0
print(f"total parameter budget fixed at {BUDGET:g} units\n")
print(f"{'members':>8} {'size each':>10} | " + " ".join(f"{'rho='+str(r):>10}" for r in (0.0, 0.3, 0.7)))
for N in (1, 2, 4, 8, 16):
    cells = [f"{simulate(BUDGET, N, rho):10.4f}" for rho in (0.0, 0.3, 0.7)]
    print(f"{N:8d} {BUDGET/N:10.1f} | " + " ".join(cells))

print("\nRead down each column for the optimal split, not just the extremes:")
print("  rho=0.0  best at N=8  -- error more than halved; split aggressively")
print("  rho=0.3  best at N=4  -- still a solid win")
print("  rho=0.7  best at N=4  -- but the win has shrunk to ~9%, and by N=16 the")
print("           bias you gave up exceeds everything you bought")
print("\nSo correlation does not simply decide 'ensemble or not' -- it decides HOW FAR")
print("you can split. High correlation pushes the optimum toward few, larger members")
print("and shrinks the prize to the point where the operational cost is not worth it.")
print("\nThe practical reading: 'many small models beat one big model' holds only when")
print("the members are genuinely diverse. Same architecture, same data, different")
print("seeds usually lands nearer rho=0.5-0.8 than rho=0.")

total parameter budget fixed at 64 units

 members  size each |    rho=0.0    rho=0.3    rho=0.7
       1       64.0 |     1.1250     1.1250     1.1250
       2       32.0 |     0.6768     0.8268     1.0268
       4       16.0 |     0.5000     0.7250     1.0250
       8        8.0 |     0.4786     0.7411     1.0911
      16        4.0 |     0.5625     0.8438     1.2188

Read down each column for the optimal split, not just the extremes:
  rho=0.0  best at N=8  -- error more than halved; split aggressively
  rho=0.3  best at N=4  -- still a solid win
  rho=0.7  best at N=4  -- but the win has shrunk to ~9%, and by N=16 the
           bias you gave up exceeds everything you bought

So correlation does not simply decide 'ensemble or not' -- it decides HOW FAR
you can split. High correlation pushes the optimum toward few, larger members
and shrinks the prize to the point where the operational cost is not worth it.

The practical reading: 'many small models beat one big model' holds only wh

### Example 3 — averaging logits is not averaging probabilities

Both are called "ensembling". They are different operations: averaging probabilities is
an arithmetic mean, averaging logits is (up to normalisation) a **geometric** mean. They
disagree exactly when the members disagree — which is precisely the case you built the
ensemble for.

In [4]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

# Three members, one of which is confidently wrong -- the interesting case.
logits = np.array([
    [4.0, 1.0, 0.0],     # confident: class 0
    [3.5, 1.2, 0.2],     # confident: class 0
    [0.0, 6.0, 0.0],     # very confident: class 1  (the dissenter)
])
probs = softmax(logits)

avg_probs = probs.mean(axis=0)
avg_logits = softmax(logits.mean(axis=0))

print("member probabilities:")
for i, p in enumerate(probs):
    print(f"  member {i}: {np.round(p, 4)}")
print(f"\naverage of probabilities : {np.round(avg_probs, 4)}  -> class {avg_probs.argmax()}")
print(f"average of logits        : {np.round(avg_logits, 4)}  -> class {avg_logits.argmax()}")

print("\nProbability averaging is a vote: two members outrank one, class 0 wins.")
print("Logit averaging is closer to a geometric mean, so a single very confident")
print("member can veto the others -- and here it flips the prediction.")
print("\nNeither is 'correct'. Probability averaging is more robust to an overconfident")
print("member; logit averaging propagates confidence and is the right choice when your")
print("members are well calibrated. Pick deliberately -- most people do it by accident.")

member probabilities:
  member 0: [0.9362 0.0466 0.0171]
  member 1: [0.8794 0.0882 0.0324]
  member 2: [0.0025 0.9951 0.0025]

average of probabilities : [0.606  0.3766 0.0173]  -> class 0
average of logits        : [0.4254 0.5372 0.0373]  -> class 1

Probability averaging is a vote: two members outrank one, class 0 wins.
Logit averaging is closer to a geometric mean, so a single very confident
member can veto the others -- and here it flips the prediction.

Neither is 'correct'. Probability averaging is more robust to an overconfident
member; logit averaging propagates confidence and is the right choice when your
members are well calibrated. Pick deliberately -- most people do it by accident.


### Example 4 — model soups: the ensemble that costs nothing at inference

Averaging *weights* rather than outputs gives one model of the original size. It only
works when the members occupy the same loss basin — fine-tunes of a shared checkpoint,
not independent runs.

In [5]:
# A quadratic loss landscape with two distinct basins.
def loss(w, centre, scale=1.0):
    return scale * np.sum((w - centre) ** 2, axis=-1)

basin_a = np.array([1.0, 1.0])
basin_b = np.array([-3.0, 2.5])

# Same-basin members: three fine-tunes that wandered near basin A.
same = np.array([basin_a + rng.standard_normal(2) * 0.3 for _ in range(3)])
# Different-basin members: independent runs that found different minima.
diff = np.array([basin_a + rng.standard_normal(2) * 0.3,
                 basin_b + rng.standard_normal(2) * 0.3])

def landscape(w):
    '''Loss = the better of the two basins, i.e. a two-well landscape.'''
    return np.minimum(loss(w, basin_a), loss(w, basin_b))

print("SAME basin (fine-tunes of one checkpoint):")
print(f"  member losses      : {np.round([landscape(w) for w in same], 4)}")
print(f"  mean member loss   : {np.mean([landscape(w) for w in same]):.4f}")
print(f"  soup (avg weights) : {landscape(same.mean(axis=0)):.4f}   <- better than any member")

print("\nDIFFERENT basins (independent runs):")
print(f"  member losses      : {np.round([landscape(w) for w in diff], 4)}")
print(f"  mean member loss   : {np.mean([landscape(w) for w in diff]):.4f}")
print(f"  soup (avg weights) : {landscape(diff.mean(axis=0)):.4f}   <- far WORSE than either")

print("\nThe midpoint between two basins is the ridge between them, and a network whose")
print("weights sit on that ridge is broken. Averaging *outputs* would have been fine.")
print("\nSo: soup fine-tunes of a shared checkpoint (1x inference cost, some of the")
print("gain); ensemble outputs across independent runs (Nx cost, all of the gain).")

SAME basin (fine-tunes of one checkpoint):
  member losses      : [0.1375 0.0033 0.1089]
  mean member loss   : 0.0832
  soup (avg weights) : 0.0010   <- better than any member

DIFFERENT basins (independent runs):
  member losses      : [0.0241 0.154 ]
  mean member loss   : 0.0891
  soup (avg weights) : 3.6599   <- far WORSE than either

The midpoint between two basins is the ridge between them, and a network whose
weights sit on that ridge is broken. Averaging *outputs* would have been fine.

So: soup fine-tunes of a shared checkpoint (1x inference cost, some of the
gain); ensemble outputs across independent runs (Nx cost, all of the gain).


## 6. Gotchas & Pitfalls

- **Not measuring `ρ`.** Everything follows from it, and it takes one line: correlate the
  members' per-example errors on a validation set. If `ρ > 0.9` your ensemble is
  decorative.
- **Ensembling checkpoints from one run and calling it a deep ensemble.** Snapshot
  ensembles are much more correlated than independently-trained models. They are
  cheap and they help a little; they are not the same technique.
- **Counting parameters instead of latency.** `N` members is `N×` the compute — but if
  they run *in parallel* on separate devices the added **latency** is near zero while
  throughput cost is `N×`. Which one binds you changes the decision entirely.
- **Souping models from different pretrained checkpoints.** Example 4. The result is not
  a worse model, it is a broken one. Same-basin is a hard prerequisite.
- **Averaging probabilities from miscalibrated members.** An overconfident member
  dominates the average. Calibrate (temperature scaling) *before* ensembling, or average
  logits deliberately.
- **Ensembling to fix bias.** Ensembling reduces *variance*. If every member is wrong in
  the same direction — a mislabelled dataset, a missing feature — the ensemble is wrong
  in that direction with more confidence.
- **Forgetting BatchNorm statistics when souping.** Averaged weights need their
  normalisation statistics recomputed with a forward pass over training data.
- **Reporting the ensemble's accuracy against a single member of the same size** rather
  than against a single model of the same *total cost*. Example 2 is the fair comparison.

## 7. When to Use vs Alternatives

| Goal | Reach for |
|---|---|
| Best accuracy, cost no object | **Deep ensemble** of independently-trained models |
| Some of the gain, 1× inference cost | **Model soup** (same-basin members only) |
| Ensemble accuracy at 1× cost, and you can train | **Ensemble → [distil](knowledge-distillation.ipynb)** into one student |
| Calibrated uncertainty | **Deep ensemble** — still the strongest simple baseline |
| More capacity, flat inference cost | [Mixture of Experts](../08-architectures/mixture-of-experts.ipynb) — a sparse ensemble |
| Actually make the model smaller | Not this notebook: [pruning](pruning.ipynb), [distillation](knowledge-distillation.ipynb), [quantization](quantization-gptq-awq.ipynb), [low-rank](low-rank-factorization.ipynb) |

**The honest position.** Ensembling is the weakest *compression* technique in this
library and the strongest *accuracy* technique. Taken literally — deploy `N` models,
average their outputs — it makes inference strictly more expensive, and the last row of
the table is the honest answer for anyone whose actual goal is a smaller model.

Its real place in a compression pipeline is as a **teacher**. An ensemble is the cheapest
way to build a model better than anything you can train directly, and
[distillation](knowledge-distillation.ipynb) converts that into a single cheap student.
"Ensemble, then distil" is a genuinely strong recipe and it is how most of the value here
gets realised in production.

Model soups are the other exception, and an unusually good deal: if you already ran a
hyperparameter sweep from one pretrained checkpoint, averaging the good runs' weights
costs one line of code and one forward pass, and gives a free accuracy improvement at
unchanged inference cost.

## 8. Resources

- [Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles](https://arxiv.org/abs/1612.01474) — Lakshminarayanan et al.; the deep-ensembles baseline, and the calibration argument.
- [Model soups: averaging weights of multiple fine-tuned models improves accuracy without increasing inference time](https://arxiv.org/abs/2203.05482) — Example 4's technique, and the same-basin condition.
- [Snapshot Ensembles: Train 1, Get M for Free](https://arxiv.org/abs/1704.00109) — collecting members from a single cyclic-LR run.
- [Linear Mode Connectivity and the Lottery Ticket Hypothesis](https://arxiv.org/abs/1912.05671) — why same-basin weight averaging works at all.
- [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531) — Hinton et al.; section 5 is specifically about distilling an *ensemble* into one model.
- [When Ensembling Smaller Models is More Efficient than Single Large Models](https://arxiv.org/abs/2005.00570) — the Example 2 question, tested on real architectures.
- [scikit-learn: ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html) — bagging, boosting and stacking, with the classical bias/variance treatment.